# PS LiDAR - Development Playground

**Available Bricks:**
- Brick 1: Data Loading
- Brick 2: Circular Clipping (manual coordinates)
- **Brick 3.5: Noise Filtering (SOR)**  NUEVO
- Brick 3: Detección de Normalización
- Brick 4: Ground Filtering
- Brick 5: Normalización de Height
- Brick 5.5: Export Checkpoints
- Brick 6: Visualización 3D
- Brick 7: Segmentación de Árboles
- **Brick 7b: Separación de Understory** ← NUEVO

In [1]:
import os
import sys
import time
from pathlib import Path

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.core import (
    PointCloudLoader,
    detect_normalization,
    classify_ground,
    clip_circular_plot,
    normalize_heights,
    export_point_cloud,
    segment_trees,
    filter_noise_sor,
    separate_understory,
)

print("✓ Módulos importados")

✓ Módulos importados


---
## 1. Load File (Brick 1)

In [3]:
FILE_PATH = "C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/raw/HQP079_01_raw.laz"

loader = PointCloudLoader(FILE_PATH)
loader.load()

meta = loader.get_metadata()
print(f"File: {meta['filename']}")
print(f"Points: {meta['point_count']:,}")
print(f"Tamaño: {meta['file_size_mb']} MB")

Archivo: HQP079_01_raw.laz
Puntos: 19,173,291
TamaÃƒÂ±o: 209.63 MB


In [4]:
# Load XYZ y campos escalares disponibles
xyz_full = loader.get_xyz()

scalar_fields = {}
for field in ['intensity', 'return_number', 'number_of_returns', 'classification']:
    try:
        scalar_fields[field] = loader.get_attribute(field)
        print(f" {field}: {len(scalar_fields[field]):,} valores")
    except:
        print(f" {field}: no disponible")

print(f"\nMemoria XYZ: {xyz_full.nbytes / (1024**2):.1f} MB")

Ã¢Å“â€œ intensity: 19,173,291 valores
Ã¢Å“â€œ return_number: 19,173,291 valores
Ã¢Å“â€œ number_of_returns: 19,173,291 valores
Ã¢Å“â€œ classification: 19,173,291 valores

Memoria XYZ: 438.8 MB


---
## 2. Recorte Circular (Brick 2)

**Instrucciones:**
1. Open the original file in CloudCompare
2. Usar herramienta "Point Picking" para ubicar el centro del mat
3. Copiar las coordenadas Xg, Yg mostradas
4. Pegar los valores en `CENTER_X` y `CENTER_Y` abajo

In [5]:
# 
# PARÁMETROS DEL USUARIO - Modificar según el plot
# 

# Coordenadas del centro (obtenidas de CloudCompare Point Picking)
CENTER_X = -0.311372
CENTER_Y = -1.461118

# Radio del plot en metros
PLOT_RADIUS = 16.0

# 

print(f"Centro: ({CENTER_X:.6f}, {CENTER_Y:.6f})")
print(f"Radio: {PLOT_RADIUS}m")

Centro: (-0.311372, -1.461118)
Radio: 16.0m


In [6]:
# Ejecutar recorte circular
t0 = time.perf_counter()
clip_result = clip_circular_plot(xyz_full, CENTER_X, CENTER_Y, PLOT_RADIUS)
elapsed = time.perf_counter() - t0

plot_indices = clip_result.indices

print(f" Recorte en {elapsed*1000:.0f}ms")
print(f"Points originales: {len(xyz_full):,}")
print(f"Points en plot: {clip_result.n_points:,} ({clip_result.n_points/len(xyz_full):.1%})")

Ã¢Å“â€œ Recorte en 1017ms
Puntos originales: 19,173,291
Puntos en plot: 10,040,816 (52.4%)


In [7]:
# Aplicar recorte a XYZ y campos escalares
xyz = xyz_full[plot_indices]

plot_scalars = {}
for field, values in scalar_fields.items():
    plot_scalars[field] = values[plot_indices]

print(f"Plot XYZ: {xyz.shape}")
print(f"Campos escalares: {list(plot_scalars.keys())}")

# Liberar memoria
del xyz_full, scalar_fields
import gc; gc.collect()
print(" Memoria liberada")

Plot XYZ: (10040816, 3)
Campos escalares: ['intensity', 'return_number', 'number_of_returns', 'classification']
Ã¢Å“â€œ Memoria liberada


---
## 2.5 Filtering de Ruido (Brick 3.5)

In [8]:
# 
# BRICK 3.5: NOISE FILTERING (SOR)
# 

print("Aplicando filtro de ruido SOR...")
t0 = time.perf_counter()
noise_result = filter_noise_sor(xyz, k_neighbors=10, std_ratio=2.0, verbose=True)

# Reemplazar xyz con points limpios
xyz = noise_result.clean_xyz
plot_scalars = {k: v[noise_result.clean_indices] for k, v in plot_scalars.items()}

print(f"\n Filtering completed in {time.perf_counter()-t0:.1f}s")

Aplicando filtro de ruido SOR...
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
SOR filtering: 10,040,816 points, k=10, std=2.0
  Removed: 432,381 points (4.31%)
  Remaining: 9,608,435 points

Ã¢Å“â€œ Filtrado en 33.4s


---
## 3. Análisis de Normalización (Brick 3)

In [9]:
analysis = detect_normalization(xyz)
print(f"Estatus: {analysis.status.value.upper()}")
print(f"Normalised?: {analysis.is_normalized}")
print(f"Rango Z: {analysis.z_min:.2f}m a {analysis.z_max:.2f}m")

Estatus: NOT_NORMALIZED
Ã‚Â¿Normalizada?: False
Rango Z: -2.14m a 22.73m


---
## 4. Ground Filtering (Brick 4)

In [10]:
print("Ejecutando CSF...")
t0 = time.perf_counter()

ground_result = classify_ground(
    xyz,
    cloth_resolution=1.0,
    rigidness=1,
    class_threshold=0.5,
    slope_smooth=True,
)

print(f" Completed in {time.perf_counter() - t0:.2f}s")
print(f"Ground: {ground_result.n_ground:,} ({ground_result.ground_ratio:.1%})")
print(f"Vegetación: {ground_result.n_off_ground:,}")

Ejecutando CSF...
Ã¢Å“â€œ Completado en 4.21s
Suelo: 2,922,123 (30.4%)
VegetaciÃƒÂ³n: 6,686,312


In [11]:
# Separate ground and vegetation
ground_xyz = xyz[ground_result.ground_indices]
vegetation_xyz = xyz[ground_result.off_ground_indices]

ground_scalars = {k: v[ground_result.ground_indices] for k, v in plot_scalars.items()}
vegetation_scalars = {k: v[ground_result.off_ground_indices] for k, v in plot_scalars.items()}

print(f"Ground: {len(ground_xyz):,} points")
print(f"Vegetación: {len(vegetation_xyz):,} points")

Suelo: 2,922,123 puntos
VegetaciÃƒÂ³n: 6,686,312 puntos


---
## 5. Normalización de Height (Brick 5)

In [12]:
print("Normalising heights...")
t0 = time.perf_counter()

veg_norm_result = normalize_heights(vegetation_xyz, ground_xyz, resolution=0.5)
veg_normalized = veg_norm_result.xyz_normalized

ground_norm_result = normalize_heights(ground_xyz, ground_xyz, resolution=0.5)
ground_normalized = ground_norm_result.xyz_normalized

print(f" Completed in {(time.perf_counter() - t0)*1000:.0f}ms")
print(f"")
print(f"Vegetación: Z = {veg_normalized[:, 2].min():.2f}m a {veg_normalized[:, 2].max():.2f}m")
print(f"Ground: Z = {ground_normalized[:, 2].min():.2f}m a {ground_normalized[:, 2].max():.2f}m")

Normalizando alturas...
Ã¢Å“â€œ Completado en 1254ms

VegetaciÃƒÂ³n: Z = -0.70m a 25.03m
Suelo: Z = -0.69m a 0.73m


---
## 5.5 Export Checkpoints

In [ ]:
# Directorio de salida
OUTPUT_DIR = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/raw")

# Export vegetación
veg_file = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/processed/HQP079_01_veg_normalized.laz")
export_point_cloud(
    veg_file,
    veg_normalized,
    intensity=vegetation_scalars.get('intensity'),
    return_number=vegetation_scalars.get('return_number'),
    number_of_returns=vegetation_scalars.get('number_of_returns'),
    classification=vegetation_scalars.get('classification'),
)
print(f"✓ Vegetación: {veg_file.name} ({veg_file.stat().st_size / (1024**2):.1f} MB)")

# Export ground
ground_file = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/processed/HQP079_01_ground_normalized.laz")
export_point_cloud(
    ground_file,
    ground_normalized,
    intensity=ground_scalars.get('intensity'),
    return_number=ground_scalars.get('return_number'),
    number_of_returns=ground_scalars.get('number_of_returns'),
    classification=ground_scalars.get('classification'),
)
print(f" Ground: {ground_file.name} ({ground_file.stat().st_size / (1024**2):.1f} MB)")

---
## 6. Visualización 3D (Brick 6)

In [ ]:
import open3d as o3d
import numpy as np

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(veg_normalized)

# Colour by height
z = veg_normalized[:, 2]
z_scaled = (z - z.min()) / (z.max() - z.min() + 1e-6)
colors = np.zeros((len(z_scaled), 3))
colors[:, 0] = z_scaled
colors[:, 1] = 1 - np.abs(2 * z_scaled - 1)
colors[:, 2] = 1 - z_scaled
pcd.colors = o3d.utility.Vector3dVector(colors)

print(f"Nube: {len(pcd.points):,} points")

In [ ]:
o3d.visualization.draw_geometries([pcd], window_name="Normalised Vegetation", width=1280, height=720)


---
## 7. Geometrics features extration

---
#### Upload files from ceckpoint

In [1]:
# ============================================================================
# BRICK 7.1 CONTROL RUN: BASELINE VS OPTIMISED MULTISCALE FEATURES
# ============================================================================

import gc
import os
import subprocess
import sys
import time
import types
from pathlib import Path

import laspy
import numpy as np

# ----------------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------------
DEFAULT_PROJECT_ROOT = Path(r"c:/Users/geoal/Documents/SoftwareDev/PS_LiDAR")

# Baseline git ref:
# - None -> auto-detect a valid previous revision for this file.
# - Use an explicit ref (for example "HEAD~1") if you want to pin baseline.
BASELINE_REF = None

RUN_BASELINE = True       # True: compare baseline vs optimised when available
RUN_FULL_CLOUD = True     # True: run full cloud, False: quick sample
QUICK_SAMPLE_SIZE = 500_000
QUALITY_SAMPLE_SIZE = 500_000
RNG_SEED = 42

# If None, checkpoint is auto-discovered in inputs/outputs/notebooks.
# You can set an absolute path or a path relative to PROJECT_ROOT.
MANUAL_CHECKPOINT = None
SHOW_TOP_CANDIDATES = 10

MS_KWARGS = dict(
    scales=(0.10, 0.20, 0.40),
    voxel_size=0.10,
    min_neighbors=8,
    return_per_scale=False,
    verbose=True,
)

FEATURE_NAMES = [
    "verticality",
    "linearity",
    "planarity",
    "sphericity",
    "roughness",
    "mean_curvature",
    "gaussian_curvature",
    "neighbor_count",
    "surface_density",
    "volume_density",
]

# ----------------------------------------------------------------------------
# Setup imports and git context
# ----------------------------------------------------------------------------
def resolve_project_root(default_root: Path) -> Path:
    if default_root.exists() and (default_root / "src/core/multiscale_features.py").exists():
        return default_root

    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "src/core/multiscale_features.py").exists():
            return base

    raise FileNotFoundError(
        "Could not resolve project root containing src/core/multiscale_features.py"
    )


def bootstrap_import_paths(project_root: Path) -> None:
    root_str = str(project_root)
    src_str = str(project_root / "src")

    if root_str not in sys.path:
        sys.path.insert(0, root_str)
    if src_str not in sys.path:
        sys.path.insert(0, src_str)

    # Remove conflicting 'src' module loaded from another location.
    src_mod = sys.modules.get("src")
    if src_mod is not None:
        src_file = getattr(src_mod, "__file__", "") or ""
        if src_file and root_str.lower() not in src_file.lower():
            del sys.modules["src"]


PROJECT_ROOT = resolve_project_root(DEFAULT_PROJECT_ROOT)
bootstrap_import_paths(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

try:
    from src.core.multiscale_features import compute_multiscale_geometric_features as new_ms
except ModuleNotFoundError:
    from core.multiscale_features import compute_multiscale_geometric_features as new_ms


def _git_output(args):
    return subprocess.check_output(
        args,
        cwd=str(PROJECT_ROOT),
        text=True,
        encoding="utf-8",
        stderr=subprocess.STDOUT,
    )


def load_baseline_function(git_ref=None):
    rel_path = "src/core/multiscale_features.py"

    refs_to_try = []
    if git_ref:
        refs_to_try.append(git_ref)

    refs_to_try.extend(["HEAD~1", "HEAD^", "HEAD"])

    try:
        history = _git_output(
            ["git", "rev-list", "--max-count=30", "HEAD", "--", rel_path]
        ).splitlines()
        refs_to_try.extend(history[1:])
    except Exception:
        pass

    refs_unique = []
    for ref in refs_to_try:
        if ref and ref not in refs_unique:
            refs_unique.append(ref)

    tried = []
    for ref in refs_unique:
        try:
            old_src = _git_output(["git", "show", f"{ref}:{rel_path}"])
            old_mod = types.ModuleType("src.core.ms_old")
            old_mod.__package__ = "src.core"
            sys.modules["src.core.ms_old"] = old_mod
            exec(compile(old_src, "src/core/ms_old.py", "exec"), old_mod.__dict__)
            print(f"Baseline loaded from ref: {ref}")
            return old_mod.compute_multiscale_geometric_features, ref
        except Exception:
            tried.append(ref)

    raise RuntimeError(
        "Could not load a baseline version from git. "
        f"Tried refs: {tried}"
    )


def find_checkpoint(project_root, manual_checkpoint=None, top_n=10):
    if manual_checkpoint:
        p = Path(manual_checkpoint)
        if not p.is_absolute():
            p = project_root / p
        if not p.exists():
            raise FileNotFoundError(f"Manual checkpoint does not exist: {p}")
        return p

    search_roots = [
        project_root / "inputs",
        project_root / "outputs",
        project_root / "notebooks",
    ]

    candidates = []
    for root in search_roots:
        if not root.exists():
            continue
        candidates.extend(root.rglob("*.laz"))
        candidates.extend(root.rglob("*.las"))

    if not candidates:
        raise FileNotFoundError(
            f"No LAS/LAZ files found under: {[str(p) for p in search_roots]}"
        )

    def score(path_obj):
        t = str(path_obj).lower()
        s = 0
        if "inputs" in t:
            s += 200
        if "normal" in t or "normalised" in t or "normalized" in t:
            s += 120
        if "veg" in t or "vegetation" in t:
            s += 60
        if "treeiso_filtered" in t:
            s += 20
        try:
            size = path_obj.stat().st_size
        except OSError:
            size = 0
        return (s, size)

    ranked = sorted(candidates, key=score, reverse=True)
    print("Top checkpoint candidates:")
    for p in ranked[:top_n]:
        print(f"  - {p}")

    return ranked[0]


# ----------------------------------------------------------------------------
# Load checkpoint cloud
# ----------------------------------------------------------------------------
checkpoint_file = find_checkpoint(
    PROJECT_ROOT,
    manual_checkpoint=MANUAL_CHECKPOINT,
    top_n=SHOW_TOP_CANDIDATES,
)
print(f"Selected checkpoint: {checkpoint_file}")

las = laspy.read(str(checkpoint_file))
xyz_full = np.column_stack([las.x, las.y, las.z]).astype(np.float64, copy=False)

print(f"Total points: {len(xyz_full):,}")
print(f"Z range: {xyz_full[:, 2].min():.3f} m to {xyz_full[:, 2].max():.3f} m")

# ----------------------------------------------------------------------------
# Select evaluation cloud
# ----------------------------------------------------------------------------
rng = np.random.default_rng(RNG_SEED)

if RUN_FULL_CLOUD:
    xyz_eval = xyz_full
    print("Mode: full cloud")
else:
    n_quick = min(QUICK_SAMPLE_SIZE, len(xyz_full))
    idx_quick = rng.choice(len(xyz_full), size=n_quick, replace=False)
    xyz_eval = xyz_full[idx_quick]
    print(f"Mode: quick sample ({len(xyz_eval):,} points)")

n_quality = min(QUALITY_SAMPLE_SIZE, len(xyz_eval))
quality_idx = rng.choice(len(xyz_eval), size=n_quality, replace=False)
print(f"Quality comparison sample: {n_quality:,} points")

# ----------------------------------------------------------------------------
# Baseline run (optional)
# ----------------------------------------------------------------------------
baseline_enabled = bool(RUN_BASELINE)
old_sample = None
t_old = None
baseline_ref_used = None

if baseline_enabled:
    try:
        old_ms, baseline_ref_used = load_baseline_function(BASELINE_REF)
    except Exception as exc:
        baseline_enabled = False
        print("\nWarning: baseline run disabled (could not load baseline function).")
        print(f"Reason: {exc}")
        print("Continuing with optimised-only run.")

if baseline_enabled:
    print("\nRunning baseline...")
    t0 = time.perf_counter()
    res_old = old_ms(xyz_eval, **MS_KWARGS)
    t_old = time.perf_counter() - t0
    print(f"Baseline time: {t_old:.2f} s")

    old_sample = {
        name: getattr(res_old, name)[quality_idx].astype(np.float32, copy=True)
        for name in FEATURE_NAMES
    }

    del res_old
    gc.collect()

# ----------------------------------------------------------------------------
# Optimised run
# ----------------------------------------------------------------------------
print("\nRunning optimised...")
t0 = time.perf_counter()
res_new = new_ms(xyz_eval, **MS_KWARGS)
t_new = time.perf_counter() - t0
print(f"Optimised time: {t_new:.2f} s")

# ----------------------------------------------------------------------------
# KPI summary
# ----------------------------------------------------------------------------
print("\nKPI summary")
print(f"  points_evaluated: {len(xyz_eval):,}")
print(f"  optimised_time_s: {t_new:.2f}")

if baseline_enabled and t_old is not None:
    speedup_x = t_old / max(t_new, 1e-9)
    saved_s = t_old - t_new
    saved_pct = 100.0 * saved_s / max(t_old, 1e-9)

    print(f"  baseline_ref_used: {baseline_ref_used}")
    print(f"  baseline_time_s: {t_old:.2f}")
    print(f"  speedup_x: {speedup_x:.3f}")
    print(f"  time_saved_s: {saved_s:.2f}")
    print(f"  time_saved_pct: {saved_pct:.2f}%")

    print("\nQuality deltas (optimised vs baseline)")
    for name in FEATURE_NAMES:
        old_vals = old_sample[name]
        new_vals = getattr(res_new, name)[quality_idx]
        diff = np.abs(new_vals - old_vals)
        print(
            f"  {name}: "
            f"mae={np.mean(diff):.9f}, "
            f"p95={np.percentile(diff, 95):.9f}, "
            f"max={np.max(diff):.9f}"
        )
else:
    print("  baseline_time_s: not available")
    print("  speedup_x: not available")

print("\nOptimised feature means")
for name in FEATURE_NAMES:
    print(f"  {name}_mean: {float(np.mean(getattr(res_new, name))):.6f}")




Project root: c:\Users\geoal\Documents\SoftwareDev\PS_LiDAR
Top checkpoint candidates:
  - c:\Users\geoal\Documents\SoftwareDev\PS_LiDAR\outputs\treeiso\HQP079_01_veg_normalized_treeiso_filtered.laz
  - c:\Users\geoal\Documents\SoftwareDev\PS_LiDAR\outputs\treeiso\HQP079_01_veg_normalized_treeiso_tree_only.laz
  - c:\Users\geoal\Documents\SoftwareDev\PS_LiDAR\outputs\treeiso\HQP079_01_veg_normalized_treeiso_understory_only.laz
  - c:\Users\geoal\Documents\SoftwareDev\PS_LiDAR\outputs\trees.laz
  - c:\Users\geoal\Documents\SoftwareDev\PS_LiDAR\outputs\trees_02.laz
  - c:\Users\geoal\Documents\SoftwareDev\PS_LiDAR\outputs\trees_03.laz
  - c:\Users\geoal\Documents\SoftwareDev\PS_LiDAR\outputs\understory_03.laz
  - c:\Users\geoal\Documents\SoftwareDev\PS_LiDAR\outputs\understory_02.laz
  - c:\Users\geoal\Documents\SoftwareDev\PS_LiDAR\outputs\understory.laz
  - c:\Users\geoal\Documents\SoftwareDev\PS_LiDAR\outputs\ml\phase4_smoke_input.las
Selected checkpoint: c:\Users\geoal\Documents\Soft

---

In [3]:
# ==============================================================
# BRICK 7.1: MULTISCALE GEOMETRIC FEATURE EXTRACTION
# ==============================================================
from src.core import (
    compute_all_features_fast,
    compute_multiscale_geometric_features,
    estimate_local_radius,
)
import time
import numpy as np

print("Brick 7.1: extracting geometric features...")

if "veg_normalized" not in globals():
    if "xyz_full" in globals():
        veg_normalized = xyz_full
        print(f"Using xyz_full as veg_normalized ({len(veg_normalized):,} points)")
    elif "xyz_eval" in globals():
        veg_normalized = xyz_eval
        print(f"Using xyz_eval as veg_normalized ({len(veg_normalized):,} points)")
    else:
        raise NameError(
            "veg_normalized is not defined and no fallback cloud found. "
            "Run checkpoint loading first (or Brick 7.1 control run) to create xyz_full."
        )

n_points_total = len(veg_normalized)
t0_total = time.perf_counter()

t0 = time.perf_counter()
features, dist_to_ground, dist_to_top = compute_all_features_fast(
    veg_normalized,
    voxel_size=0.1,
    k_neighbors=20,
    verbose=True,
)
t_features_basic = time.perf_counter() - t0

t0 = time.perf_counter()
ms_features = compute_multiscale_geometric_features(
    veg_normalized,
    scales=(0.10, 0.20, 0.40),
    voxel_size=0.10,
    min_neighbors=8,
    return_per_scale=False,
    verbose=True,
)
t_features_multiscale = time.perf_counter() - t0

print("Estimating local stem radius...")
t0 = time.perf_counter()
local_radius = estimate_local_radius(veg_normalized, verbose=True)
t_local_radius = time.perf_counter() - t0

t_total = time.perf_counter() - t0_total

print("Brick 7.1 KPI summary")
print(f"  n_points_total: {n_points_total:,}")
print(f"  t_features_basic_s: {t_features_basic:.2f}")
print(f"  t_features_multiscale_s: {t_features_multiscale:.2f}")
print(f"  t_local_radius_s: {t_local_radius:.2f}")
print(f"  t_total_s: {t_total:.2f}")

print("  feature_stats:")
print(f"    verticality_mean: {np.mean(features.verticality):.4f}")
print(f"    verticality_p95: {np.percentile(features.verticality, 95):.4f}")
print(f"    linearity_mean: {np.mean(features.linearity):.4f}")
print(f"    linearity_p95: {np.percentile(features.linearity, 95):.4f}")
print(f"    sphericity_mean: {np.mean(features.sphericity):.4f}")
print(f"    sphericity_p95: {np.percentile(features.sphericity, 95):.4f}")
print(f"    roughness_mean: {np.mean(ms_features.roughness):.4f}")
print(f"    roughness_p95: {np.percentile(ms_features.roughness, 95):.4f}")

print("  radius_stats:")
print(f"    local_radius_median: {np.median(local_radius):.4f}")
print(f"    local_radius_p95: {np.percentile(local_radius, 95):.4f}")




Brick 7.1: extracting geometric features...
Using xyz_full as veg_normalized (6,686,312 points)
Computing geometric features (optimized) for 6,686,312 points...
  Voxel size: 0.1m, K-neighbors: 20
  Voxelized: 6,686,312 points → 1,146,011 voxels (17.1%)
  Computing PCA on 1,146,011 voxel centroids...
    Processed 100,000 / 1,146,011 voxels (8.7%)
    Processed 200,000 / 1,146,011 voxels (17.5%)
    Processed 300,000 / 1,146,011 voxels (26.2%)
    Processed 400,000 / 1,146,011 voxels (34.9%)
    Processed 500,000 / 1,146,011 voxels (43.6%)
    Processed 600,000 / 1,146,011 voxels (52.4%)
    Processed 700,000 / 1,146,011 voxels (61.1%)
    Processed 800,000 / 1,146,011 voxels (69.8%)
    Processed 900,000 / 1,146,011 voxels (78.5%)
    Processed 1,000,000 / 1,146,011 voxels (87.3%)
    Processed 1,100,000 / 1,146,011 voxels (96.0%)
  Re-projecting features to 6,686,312 original points...
  ✓ Feature computation complete (optimized)
    Verticality: min=0.00, max=1.00, mean=0.53
    Lin

### Brick 7.2a: Stem Seeds and Connected Components

This step creates `component_labels` using strict stem seeds and voxel connected components.

### Brick 7.2b: Component Geometric Aggregation and Rule Filtering

Run this step after Brick 7.2a to compute component KPIs and apply deterministic geometric rules.

In [4]:
# ==============================================================
# BRICK 7.2A: STEM SEEDS + CONNECTED COMPONENTS
# ==============================================================
import sys
from pathlib import Path


def _resolve_project_root_for_brick(default_root: Path) -> Path:
    if default_root.exists() and (default_root / "src/core/__init__.py").exists():
        return default_root
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "src/core/__init__.py").exists():
            return base
    raise FileNotFoundError("Could not resolve project root containing src/core")


def _bootstrap_paths_for_brick(project_root: Path) -> None:
    root_str = str(project_root)
    src_str = str(project_root / "src")
    if root_str not in sys.path:
        sys.path.insert(0, root_str)
    if src_str not in sys.path:
        sys.path.insert(0, src_str)


PROJECT_ROOT = globals().get(
    "PROJECT_ROOT",
    _resolve_project_root_for_brick(Path(r"c:/Users/geoal/Documents/SoftwareDev/PS_LiDAR")),
)
_bootstrap_paths_for_brick(PROJECT_ROOT)

try:
    from src.core import extract_stem_seed_mask, label_stem_seed_components
except ModuleNotFoundError:
    from core import extract_stem_seed_mask, label_stem_seed_components

print("Brick 7.2a: extracting stem seeds and connected components...")

# Resolve Brick 7.1 outputs when this kernel session does not have the
# original variable names in memory.
if "veg_normalized" not in globals():
    if "xyz_full" in globals():
        veg_normalized = xyz_full
        print("Using 'xyz_full' as 'veg_normalized'.")
    elif "xyz_eval" in globals():
        veg_normalized = xyz_eval
        print("Using 'xyz_eval' as 'veg_normalized'.")
    else:
        raise NameError(
            "veg_normalized is not defined. Run Brick 7.1 before Brick 7.2a."
        )

if "ms_features" not in globals():
    if "res_new" in globals() and hasattr(res_new, "verticality"):
        if len(res_new.verticality) == len(veg_normalized):
            ms_features = res_new
            print("Using 'res_new' as 'ms_features'.")
        else:
            raise ValueError(
                "res_new length does not match veg_normalized. "
                "Re-run Brick 7.1 with the same point cloud before Brick 7.2a."
            )
    else:
        raise NameError(
            "ms_features is not defined. Run Brick 7.1 before Brick 7.2a."
        )

if len(ms_features.verticality) != len(veg_normalized):
    raise ValueError(
        "ms_features and veg_normalized have different lengths. "
        "Re-run Brick 7.1 with consistent inputs."
    )

stem_seed_result = extract_stem_seed_mask(
    xyz=veg_normalized,
    features=ms_features,
    z_min_stem=1.5,
    z_max_stem=4.5,
    min_verticality=0.70,
    min_linearity=0.30,
    max_sphericity=0.55,
    max_roughness=0.22,
    min_surface_density=10.0,
    min_volume_density=20.0,
    verbose=True,
)

stem_component_result = label_stem_seed_components(
    xyz=veg_normalized,
    seed_mask=stem_seed_result.seed_mask,
    voxel_size=0.10,
    min_component_points=100,
    verbose=True,
)

component_labels = stem_component_result.component_labels

seed_ratio = 100.0 * stem_seed_result.n_seeds / max(stem_seed_result.n_points, 1)
labelled_points = int((component_labels >= 0).sum())
labelled_ratio = 100.0 * labelled_points / max(stem_seed_result.n_points, 1)

print("Brick 7.2a KPI summary")
print(f"  n_points_total: {stem_seed_result.n_points:,}")
print(f"  n_stripe_points: {stem_seed_result.n_stripe:,}")
print(f"  n_seed_points: {stem_seed_result.n_seeds:,} ({seed_ratio:.2f}%)")
print(f"  n_components_raw: {stem_component_result.n_components_raw:,}")
print(f"  n_components_kept: {stem_component_result.n_components_kept:,}")
print(f"  points_in_kept_components: {labelled_points:,} ({labelled_ratio:.2f}%)")



Brick 7.2a: extracting stem seeds and connected components...
Stem seed extraction
  points: 6,686,312
  stripe points: 646,065
  seed points: 454,388
Stem component labelling
  seed points: 454,388
  seed voxels: 18,474
  raw components: 409
  kept components: 102
  removed small components: 307
Brick 7.2a KPI summary
  n_points_total: 6,686,312
  n_stripe_points: 646,065
  n_seed_points: 454,388 (6.80%)
  n_components_raw: 409
  n_components_kept: 102
  points_in_kept_components: 448,680 (6.71%)


In [5]:
# ==============================================================
# BRICK 7.2B: COMPONENT GEOMETRIC AGGREGATION AND RULE FILTERING
# ==============================================================
import sys
from collections import Counter
from pathlib import Path


def _resolve_project_root_for_brick(default_root: Path) -> Path:
    if default_root.exists() and (default_root / "src/core/__init__.py").exists():
        return default_root
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "src/core/__init__.py").exists():
            return base
    raise FileNotFoundError("Could not resolve project root containing src/core")


def _bootstrap_paths_for_brick(project_root: Path) -> None:
    root_str = str(project_root)
    src_str = str(project_root / "src")
    if root_str not in sys.path:
        sys.path.insert(0, root_str)
    if src_str not in sys.path:
        sys.path.insert(0, src_str)


PROJECT_ROOT = globals().get(
    "PROJECT_ROOT",
    _resolve_project_root_for_brick(Path(r"c:/Users/geoal/Documents/SoftwareDev/PS_LiDAR")),
)
_bootstrap_paths_for_brick(PROJECT_ROOT)

try:
    from src.core import (
        ComponentGeometricRules,
        compute_component_feature_table,
        filter_components_by_geometric_rules,
        component_ids_to_point_mask,
    )
except ModuleNotFoundError:
    from core import (
        ComponentGeometricRules,
        compute_component_feature_table,
        filter_components_by_geometric_rules,
        component_ids_to_point_mask,
    )

component_feature_table = None
component_filter_result = None
component_tree_mask = None

if "component_labels" in globals():
    labels_for_components = component_labels
    print("Using 'component_labels' from Brick 7.2a.")
elif "seg_result" in globals():
    labels_for_components = seg_result.tree_ids
    print("Using 'seg_result.tree_ids' as component labels.")
else:
    labels_for_components = None
    print("Component labels are not available yet.")
    print("Run Brick 7.2a (stem seeds + connected components), then re-run this cell.")

if labels_for_components is not None:
    component_feature_table = compute_component_feature_table(
        xyz=veg_normalized,
        component_labels=labels_for_components,
        features=ms_features,
        ignore_label=-1,
    )

    component_rules = ComponentGeometricRules(
        min_points=100,
        min_height=2.0,
        min_verticality_mean=0.30,
        min_linearity_mean=0.20,
        max_sphericity_mean=0.60,
        max_roughness_mean=0.25,
        max_mean_curvature_mean=2.00,
        max_gaussian_curvature_mean=4.00,
        min_surface_density_mean=10.0,
        min_volume_density_mean=20.0,
    )

    component_filter_result = filter_components_by_geometric_rules(
        component_feature_table,
        rules=component_rules,
    )
    component_tree_mask = component_ids_to_point_mask(
        labels_for_components,
        component_filter_result.kept_component_ids,
    )

    n_components_total = len(component_feature_table.component_ids)
    n_components_kept = len(component_filter_result.kept_component_ids)
    n_components_rejected = len(component_filter_result.rejected_component_ids)

    points_total = len(veg_normalized)
    points_kept = int(component_tree_mask.sum())

    reason_counter = Counter()
    for reasons in component_filter_result.rejection_reasons.values():
        reason_counter.update(reasons)

    print("Brick 7.2b KPI summary")
    print(f"  n_components_total: {n_components_total:,}")
    print(f"  n_components_kept: {n_components_kept:,}")
    print(f"  n_components_rejected: {n_components_rejected:,}")
    print(f"  kept_component_ratio: {100.0 * n_components_kept / max(n_components_total, 1):.2f}%")
    print(f"  points_in_kept_components: {points_kept:,} / {points_total:,} ({100.0 * points_kept / max(points_total, 1):.2f}%)")

    if reason_counter:
        print("  top_rejection_reasons:")
        for reason, count in reason_counter.most_common(5):
            print(f"    - {reason}: {count}")



Using 'component_labels' from Brick 7.2a.
Brick 7.2b KPI summary
  n_components_total: 102
  n_components_kept: 39
  n_components_rejected: 63
  kept_component_ratio: 38.24%
  points_in_kept_components: 425,450 / 6,686,312 (6.36%)
  top_rejection_reasons:
    - min_height: 63


In [9]:
# ==============================================================
# BRICK 7.3: TRUNK-CONNECTED EXPANSION + CANOPY PROTECTION
# ==============================================================
import numpy as np
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

print("Brick 7.3: expanding from component_tree_mask with canopy protection...")

# Resolve required inputs from previous bricks.
if "veg_normalized" not in globals():
    if "xyz_full" in globals():
        veg_normalized = xyz_full
        print("Using 'xyz_full' as 'veg_normalized'.")
    elif "xyz_eval" in globals():
        veg_normalized = xyz_eval
        print("Using 'xyz_eval' as 'veg_normalized'.")
    else:
        raise NameError("veg_normalized is not defined. Run Brick 7.1 first.")

if "ms_features" not in globals():
    if "res_new" in globals() and hasattr(res_new, "verticality"):
        if len(res_new.verticality) == len(veg_normalized):
            ms_features = res_new
            print("Using 'res_new' as 'ms_features'.")
        else:
            raise ValueError(
                "res_new length does not match veg_normalized. Run Brick 7.1 again."
            )
    else:
        raise NameError("ms_features is not defined. Run Brick 7.1 first.")

if "component_tree_mask" not in globals():
    raise NameError("component_tree_mask is not defined. Run Brick 7.2b first.")

if len(component_tree_mask) != len(veg_normalized):
    raise ValueError("component_tree_mask length does not match veg_normalized.")

# --- Parameters for this brick ---
expansion_voxel_size = 0.10
max_seed_xy_distance = 2.5

xyz = veg_normalized
z = xyz[:, 2]

# Dynamic canopy protection tuned to upper strata only.
canopy_protection_height = max(14.0, float(np.percentile(z, 90)))

# 1) Canopy protection: keep high points as tree at fusion stage, not in graph.
canopy_protected_mask = z >= canopy_protection_height

if not np.any(component_tree_mask):
    raise RuntimeError("component_tree_mask has no seed points for Brick 7.3 expansion.")

# 2) XY proximity gate to prevent long-distance bridging through loose vegetation.
seed_xy = xyz[component_tree_mask, :2]
seed_tree_xy = cKDTree(seed_xy)
dist_to_seed_xy, _ = seed_tree_xy.query(xyz[:, :2], k=1, workers=-1)
near_seed_xy_mask = dist_to_seed_xy <= max_seed_xy_distance

# 3) Candidate mask for upward expansion (stricter than previous run).
candidate_mask = (
    (ms_features.verticality >= 0.30)
    & (ms_features.linearity >= 0.18)
    & (ms_features.sphericity <= 0.72)
    & (ms_features.roughness <= 0.18)
    & (ms_features.surface_density >= 10.0)
    & (z >= 1.0)
    & (z < canopy_protection_height)
    & near_seed_xy_mask
)

# Ensure all high-confidence trunk components are always in the graph.
candidate_mask |= component_tree_mask

candidate_indices = np.where(candidate_mask)[0]
candidate_xyz = xyz[candidate_mask]

if len(candidate_indices) == 0:
    raise RuntimeError("No candidate points available for Brick 7.3 expansion.")

# 4) Voxelise candidates and build 26-neighbour connectivity graph.
voxel_idx = np.floor(candidate_xyz / expansion_voxel_size).astype(np.int32)
unique_voxels, inverse = np.unique(voxel_idx, axis=0, return_inverse=True)
n_voxels = len(unique_voxels)

voxel_to_id = {tuple(v): i for i, v in enumerate(unique_voxels)}

offsets = []
for dx in (-1, 0, 1):
    for dy in (-1, 0, 1):
        for dz in (-1, 0, 1):
            if dx != 0 or dy != 0 or dz != 0:
                offsets.append((dx, dy, dz))

rows = []
cols = []
for i, voxel in enumerate(unique_voxels):
    vx, vy, vz = int(voxel[0]), int(voxel[1]), int(voxel[2])
    for dx, dy, dz in offsets:
        j = voxel_to_id.get((vx + dx, vy + dy, vz + dz))
        if j is not None:
            rows.append(i)
            cols.append(j)

if rows:
    adjacency = csr_matrix(
        (np.ones(len(rows), dtype=np.int8), (rows, cols)),
        shape=(n_voxels, n_voxels),
    )
    n_components, voxel_labels = connected_components(
        adjacency,
        directed=False,
        return_labels=True,
    )
else:
    n_components = n_voxels
    voxel_labels = np.arange(n_voxels, dtype=np.int32)

# 5) Keep candidate components connected to trunk seeds only.
candidate_seed_mask = component_tree_mask[candidate_mask]
seed_voxel_ids = np.unique(inverse[candidate_seed_mask])
seed_component_ids = np.unique(voxel_labels[seed_voxel_ids])

connected_component_mask = np.isin(voxel_labels, seed_component_ids)
connected_point_mask = connected_component_mask[inverse]

expanded_connected_mask = np.zeros(len(xyz), dtype=bool)
expanded_connected_mask[candidate_indices] = connected_point_mask

# 6) Final tree mask: connected expansion plus protected canopy.
tree_final_mask = expanded_connected_mask | canopy_protected_mask
understory_final_mask = ~tree_final_mask

# Keep aliases for downstream cells.
tree_only_mask = tree_final_mask

# 7) Decision source map (0=understory, 1=component seed, 2=expanded connected, 3=canopy protected)
decision_source = np.zeros(len(xyz), dtype=np.uint8)
decision_source[expanded_connected_mask] = 2
decision_source[component_tree_mask] = 1
decision_source[canopy_protected_mask] = 3

# 8) KPIs
n_total = len(xyz)
n_seed = int(np.sum(component_tree_mask))
n_near_seed_xy = int(np.sum(near_seed_xy_mask))
n_candidate = int(np.sum(candidate_mask))
n_canopy = int(np.sum(canopy_protected_mask))
n_tree = int(np.sum(tree_final_mask))
n_under = int(np.sum(understory_final_mask))

print("Brick 7.3 KPI summary")
print(f"  n_points_total: {n_total:,}")
print(f"  canopy_protection_height_m: {canopy_protection_height:.2f}")
print(f"  max_seed_xy_distance_m: {max_seed_xy_distance:.2f}")
print(f"  n_seed_points_input: {n_seed:,} ({100.0 * n_seed / max(n_total, 1):.2f}%)")
print(f"  n_near_seed_xy: {n_near_seed_xy:,} ({100.0 * n_near_seed_xy / max(n_total, 1):.2f}%)")
print(f"  n_candidate_points: {n_candidate:,} ({100.0 * n_candidate / max(n_total, 1):.2f}%)")
print(f"  n_canopy_protected: {n_canopy:,} ({100.0 * n_canopy / max(n_total, 1):.2f}%)")
print(f"  n_tree_final: {n_tree:,} ({100.0 * n_tree / max(n_total, 1):.2f}%)")
print(f"  n_understory_final: {n_under:,} ({100.0 * n_under / max(n_total, 1):.2f}%)")
print(f"  voxel_components_total: {n_components:,}")
print(f"  voxel_components_connected_to_seed: {len(seed_component_ids):,}")



Brick 7.3: expanding from component_tree_mask with canopy protection...
Brick 7.3 KPI summary
  n_points_total: 6,686,312
  canopy_protection_height_m: 19.48
  max_seed_xy_distance_m: 2.50
  n_seed_points_input: 425,450 (6.36%)
  n_near_seed_xy: 5,870,805 (87.80%)
  n_candidate_points: 3,580,090 (53.54%)
  n_canopy_protected: 668,965 (10.00%)
  n_tree_final: 4,117,583 (61.58%)
  n_understory_final: 2,568,729 (38.42%)
  voxel_components_total: 1,475
  voxel_components_connected_to_seed: 4


### Brick 7.4: Topological Cleanup, Directed Recovery, and Export

This step refines `component_tree_mask` expansion by removing floating components, recovering near-stem false negatives, and exporting deterministic outputs.


In [ ]:
# ==============================================================
# BRICK 7.4: TOPOLOGICAL CLEANUP + DIRECTED RECOVERY + EXPORT
# ==============================================================
import numpy as np
import laspy
from pathlib import Path
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

print("Brick 7.4: applying topological cleanup and directed recovery...")

# Required inputs from previous bricks.
if "veg_normalized" not in globals():
    raise NameError("veg_normalized is not defined. Run Brick 7.1 first.")
if "component_tree_mask" not in globals():
    raise NameError("component_tree_mask is not defined. Run Brick 7.2b first.")
if "tree_final_mask" not in globals():
    raise NameError("tree_final_mask is not defined. Run Brick 7.3 first.")
if "ms_features" not in globals():
    raise NameError("ms_features is not defined. Run Brick 7.1 first.")

xyz = veg_normalized
n_points = len(xyz)
z = xyz[:, 2]

if len(component_tree_mask) != n_points:
    raise ValueError("component_tree_mask length does not match veg_normalized.")
if len(tree_final_mask) != n_points:
    raise ValueError("tree_final_mask length does not match veg_normalized.")
if len(ms_features.verticality) != n_points:
    raise ValueError("ms_features length does not match veg_normalized.")

if "canopy_protected_mask" not in globals():
    canopy_protection_height = max(14.0, float(np.percentile(z, 90)))
    canopy_protected_mask = z >= canopy_protection_height
    print(f"Generated canopy_protected_mask from z >= {canopy_protection_height:.2f}m")

# --- Parameters ---
cleanup_voxel_size = 0.10
cleanup_min_component_points = 200
recovery_xy_radius = 0.45
recovery_min_verticality = 0.22
recovery_min_linearity = 0.10
recovery_max_sphericity = 0.85
recovery_max_roughness = 0.24

# 1) Keep non-canopy tree components only if connected to seed trunk components.
base_tree_mask = tree_final_mask & ~canopy_protected_mask
base_indices = np.where(base_tree_mask)[0]

tree_connected_mask = np.zeros(n_points, dtype=bool)
n_base_components = 0
n_seed_connected_components = 0

if len(base_indices) > 0:
    base_xyz = xyz[base_tree_mask]

    voxel_idx = np.floor(base_xyz / cleanup_voxel_size).astype(np.int32)
    unique_voxels, inverse = np.unique(voxel_idx, axis=0, return_inverse=True)
    n_voxels = len(unique_voxels)

    voxel_to_id = {tuple(v): i for i, v in enumerate(unique_voxels)}

    offsets = []
    for dx in (-1, 0, 1):
        for dy in (-1, 0, 1):
            for dz in (-1, 0, 1):
                if dx != 0 or dy != 0 or dz != 0:
                    offsets.append((dx, dy, dz))

    rows = []
    cols = []
    for i, voxel in enumerate(unique_voxels):
        vx, vy, vz = int(voxel[0]), int(voxel[1]), int(voxel[2])
        for dx, dy, dz in offsets:
            j = voxel_to_id.get((vx + dx, vy + dy, vz + dz))
            if j is not None:
                rows.append(i)
                cols.append(j)

    if rows:
        adjacency = csr_matrix(
            (np.ones(len(rows), dtype=np.int8), (rows, cols)),
            shape=(n_voxels, n_voxels),
        )
        n_base_components, voxel_labels = connected_components(
            adjacency,
            directed=False,
            return_labels=True,
        )
    else:
        n_base_components = n_voxels
        voxel_labels = np.arange(n_voxels, dtype=np.int32)

    point_component_labels = voxel_labels[inverse]

    component_ids, component_counts = np.unique(point_component_labels, return_counts=True)
    size_valid_ids = component_ids[component_counts >= cleanup_min_component_points]

    seed_in_base = component_tree_mask[base_tree_mask]
    if np.any(seed_in_base):
        seed_component_ids = np.unique(point_component_labels[seed_in_base])
    else:
        seed_component_ids = np.array([], dtype=np.int32)

    n_seed_connected_components = int(len(seed_component_ids))

    keep_point_mask = np.isin(point_component_labels, seed_component_ids)
    keep_point_mask &= np.isin(point_component_labels, size_valid_ids)

    tree_connected_mask[base_indices] = keep_point_mask

# 2) Directed recovery: restore likely false negatives close to trunk seeds.
seed_xy = xyz[component_tree_mask, :2]
if len(seed_xy) == 0:
    near_seed_xy_mask = np.zeros(n_points, dtype=bool)
else:
    seed_tree = cKDTree(seed_xy)
    nearest_seed_xy_dist, _ = seed_tree.query(xyz[:, :2], k=1, workers=-1)
    near_seed_xy_mask = nearest_seed_xy_dist <= recovery_xy_radius

understory_after_cleanup = ~(tree_connected_mask | canopy_protected_mask)

recovery_mask = (
    understory_after_cleanup
    & near_seed_xy_mask
    & (z >= 1.0)
    & (z < float(np.percentile(z, 90)))
    & (
        (ms_features.verticality >= recovery_min_verticality)
        | (ms_features.linearity >= recovery_min_linearity)
    )
    & (ms_features.sphericity <= recovery_max_sphericity)
    & (ms_features.roughness <= recovery_max_roughness)
)

# 3) Final post-rule masks.
tree_final_pre = np.array(tree_final_mask, copy=True)
tree_final_post = tree_connected_mask | canopy_protected_mask | recovery_mask
understory_final_post = ~tree_final_post

# Keep canonical variables for downstream bricks.
tree_final_mask = tree_final_post
understory_final_mask = understory_final_post
tree_only_mask = tree_final_post

# 4) Decision source update (0=understory, 1=seed, 2=connected, 3=canopy protected, 4=recovered)
decision_source_post = np.zeros(n_points, dtype=np.uint8)
decision_source_post[tree_connected_mask] = 2
decision_source_post[component_tree_mask] = 1
decision_source_post[canopy_protected_mask] = 3
decision_source_post[recovery_mask] = 4
decision_source = decision_source_post

# 5) Export outputs.
project_root_local = Path(globals().get("PROJECT_ROOT", Path.cwd()))
outputs_dir = project_root_local / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

las_source = None
if "las" in globals():
    try:
        if len(las.x) == n_points:
            las_source = las
    except Exception:
        las_source = None

if las_source is None and "checkpoint_file" in globals():
    cp = Path(str(checkpoint_file))
    if cp.exists():
        las_tmp = laspy.read(str(cp))
        if len(las_tmp.x) == n_points:
            las_source = las_tmp

if las_source is None:
    raise RuntimeError(
        "Could not resolve a LAS/LAZ source matching veg_normalized for export. "
        "Ensure 'las' or 'checkpoint_file' is available in memory."
    )

tree_out = outputs_dir / "tree_only.laz"
understory_out = outputs_dir / "understory.laz"
decision_out = outputs_dir / "decision_source.npy"

las_tree = las_source[tree_final_post]
las_under = las_source[understory_final_post]
las_tree.write(str(tree_out))
las_under.write(str(understory_out))
np.save(decision_out, decision_source_post)

# 6) KPIs.
n_tree_pre = int(np.sum(tree_final_pre))
n_tree_post = int(np.sum(tree_final_post))
n_under_post = int(np.sum(understory_final_post))
n_recovered = int(np.sum(recovery_mask))
n_canopy = int(np.sum(canopy_protected_mask))

print("Brick 7.4 KPI summary")
print(f"  n_points_total: {n_points:,}")
print(f"  n_base_components: {n_base_components:,}")
print(f"  n_seed_connected_components: {n_seed_connected_components:,}")
print(f"  n_canopy_protected: {n_canopy:,} ({100.0 * n_canopy / max(n_points, 1):.2f}%)")
print(f"  n_recovered_points: {n_recovered:,} ({100.0 * n_recovered / max(n_points, 1):.2f}%)")
print(f"  n_tree_final_post: {n_tree_post:,} ({100.0 * n_tree_post / max(n_points, 1):.2f}%)")
print(f"  n_understory_final_post: {n_under_post:,} ({100.0 * n_under_post / max(n_points, 1):.2f}%)")
print("Exports")
print(f"  tree_only: {tree_out}")
print(f"  understory: {understory_out}")
print(f"  decision_source: {decision_out}")




Brick 7.4: applying topological cleanup and directed recovery...
Brick 7.4 KPI summary
  n_points_total: 6,686,312
  n_base_components: 4
  n_seed_connected_components: 4
  n_canopy_protected: 668,965 (10.00%)
  n_recovered_points: 81,525 (1.22%)
  n_tree_final_post: 4,199,108 (62.80%)
  n_understory_final_post: 2,487,204 (37.20%)
Exports
  tree_only: c:\Users\geoal\Documents\SoftwareDev\PS_LiDAR\outputs\tree_only.laz
  understory: c:\Users\geoal\Documents\SoftwareDev\PS_LiDAR\outputs\understory.laz
  decision_source: c:\Users\geoal\Documents\SoftwareDev\PS_LiDAR\outputs\decision_source.npy


: 

---
#### Export checkpoint: Trees & Understory (Separated files)

In [4]:
# ============================================================================
# Phase 3: Exportación
# ============================================================================
import laspy

# Use is_tree_final from Phase 2 (slice filter)
trees_xyz = veg_normalized[is_tree_final]
understory_xyz = veg_normalized[~is_tree_final]

# --- Export ÁRBOLES ---
las_trees = laspy.create(file_version="1.4", point_format=6)
las_trees.x = trees_xyz[:, 0]
las_trees.y = trees_xyz[:, 1]
las_trees.z = trees_xyz[:, 2]
las_trees.add_extra_dim(laspy.ExtraBytesParams(name="verticality", type="float32"))
las_trees.add_extra_dim(laspy.ExtraBytesParams(name="linearity", type="float32"))
las_trees.add_extra_dim(laspy.ExtraBytesParams(name="sphericity", type="float32"))
las_trees.verticality = features.verticality[is_tree_final]
las_trees.linearity = features.linearity[is_tree_final]
las_trees.sphericity = features.sphericity[is_tree_final]
las_trees.write("trees_03.laz")
print(f"Exported trees_03.laz: {len(trees_xyz):,} points")

# --- Export UNDERSTORY ---
las_under = laspy.create(file_version="1.4", point_format=6)
las_under.x = understory_xyz[:, 0]
las_under.y = understory_xyz[:, 1]
las_under.z = understory_xyz[:, 2]
las_under.add_extra_dim(laspy.ExtraBytesParams(name="verticality", type="float32"))
las_under.add_extra_dim(laspy.ExtraBytesParams(name="linearity", type="float32"))
las_under.add_extra_dim(laspy.ExtraBytesParams(name="sphericity", type="float32"))
las_under.verticality = features.verticality[~is_tree_final]
las_under.linearity = features.linearity[~is_tree_final]
las_under.sphericity = features.sphericity[~is_tree_final]
las_under.write("understory_03.laz")
print(f"Exported understory_03.laz: {len(understory_xyz):,} points")

Exported trees_03.laz: 5,895,714 points
Exported understory_03.laz: 790,598 points


---
---
# BRICK 7: Segmentación de Árboles

**Option A:** Continue from Brick 5 (if you already ran everything above)  
**Option B:** Load normalised vegetation checkpoint (if you restart the kernel)


### Opción B: Load from Checkpoint

Run this cell ONLY if you restarted the kernel and want to continue from the checkpoint.

In [ ]:
# 
# Install pgeof from inside the notebook
# 

import sys
print(f"Python: {sys.executable}")
!{sys.executable} -m pip install pgeof

In [ ]:
# ==============================================================
# LOAD FROM CHECKPOINT (standalone, includes required imports)
# ==============================================================
import sys
import time
from pathlib import Path

import laspy
import numpy as np

# Resolve project root robustly (works from repo root or notebooks/)
project_root = Path.cwd()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.core import segment_trees, separate_understory

# Candidate filtered/normalised checkpoints (first existing file is used)
checkpoint_candidates = [
    project_root / "inputs/HQP079_01_veg_filtered_01.laz",
    project_root / "inputs/vegetation_normalized.laz",
    project_root / "inputs/vegetation_normalized.las",
    project_root / "outputs/HQP079_01_veg_filtered_01.laz",
    project_root / "outputs/trees.laz",
    project_root / "outputs/trees_02.laz",
    project_root / "outputs/trees_03.laz",
]

inputs_dir = project_root / "inputs"
if inputs_dir.exists():
    discovered_inputs = sorted(list(inputs_dir.glob("*.laz")) + list(inputs_dir.glob("*.las")))
    checkpoint_candidates.extend(discovered_inputs)

checkpoint_file = next((p for p in checkpoint_candidates if p.exists()), None)
if checkpoint_file is None:
    raise FileNotFoundError(
        "Could not find a filtered/normalised checkpoint. "
        f"Checked: {[str(p) for p in checkpoint_candidates]}"
    )

print(f"Loading: {checkpoint_file.name}")
las = laspy.read(str(checkpoint_file))
veg_normalized = np.column_stack([las.x, las.y, las.z])
print(f"Loaded {len(veg_normalized):,} points")
print(f"Z range: {veg_normalized[:, 2].min():.2f} m to {veg_normalized[:, 2].max():.2f} m")
print("segment_trees imported")



### 7.1 Segmentación de Árboles

In [24]:
# 
# PARÁMETROS DE SEGMENTACIÓN
# 

VOXEL_RESOLUTION = 0.05       # Resolución de voxelización (metros)
STRIPE_Z_MIN = 2.0          # Minimum height for stem detection
STRIPE_Z_MAX = 7.0            # Maximum height for stem detection
VERTICALITY_THRESHOLD = 0.7   # Umbral de verticalidad (0-1)
MAX_AXIS_DISTANCE = 2.0       # Maximum axis distance for assignment

# 

In [ ]:
print("Segmentando árboles...")
t0 = time.perf_counter()

seg_result = segment_trees(
    veg_for_segmentation,
    voxel_resolution=VOXEL_RESOLUTION,
    stripe_z_min=STRIPE_Z_MIN,
    stripe_z_max=STRIPE_Z_MAX,
    verticality_threshold=VERTICALITY_THRESHOLD,
    max_axis_distance=MAX_AXIS_DISTANCE,
    verbose=True
)

elapsed = time.perf_counter() - t0
print(f"\n Completed in {elapsed:.1f}s")
print(f"Árboles detectados: {seg_result.n_trees}")
print(f"Points asignados: {len(veg_for_segmentation) - seg_result.unassigned_count:,}")
print(f"Points sin asignar: {seg_result.unassigned_count:,}")

In [ ]:
# Tree summary
print("\n=== Tree Summary ===")
print(f"{'ID':>4} {'Points':>12} {'Max Height':>12} {'Axis Dev.':>10}")
print("-" * 42)
for info in seg_result.tree_info:
    print(f"{info.tree_id:>4} {info.n_points:>12,} {info.height_max:>10.1f}m {info.axis_deviation_deg:>9.1f}°")

### 7.2 Export with tree_id

In [ ]:
# Export segmented cloud with tree_id as scalar field
import laspy

OUTPUT_DIR = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/raw")
seg_file = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/processed/HQP079_01_tree_segmented_2.laz")

# Create LAS file
header = laspy.LasHeader(version="1.4", point_format=0)
las_out = laspy.LasData(header)

las_out.x = veg_for_segmentation[:, 0]
las_out.y = veg_for_segmentation[:, 1]
las_out.z = veg_for_segmentation[:, 2]

# Agregar tree_id como campo extra
las_out.add_extra_dim(laspy.ExtraBytesParams(name="tree_id", type="int32", description="Tree ID"))
las_out.tree_id = seg_result.tree_ids

las_out.write(str(seg_file))
print(f" Exportado: {seg_file.name} ({seg_file.stat().st_size / (1024**2):.1f} MB)")
print(f"  Campo escalar 'tree_id' incluido para visualizar en CloudCompare")

In [ ]:
# 
# 7.2b CHECKPOINT - Export locations de árboles
# 
import os
import sys
import importlib
import numpy as np
import laspy
from pathlib import Path

# Agregar módulo al path
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

# Reload module to get the updated version
import src.core.segmentation as seg_module
importlib.reload(seg_module)
from src.core.segmentation import export_tree_locations, TreeSegmentationResult, TreeInfo

# Load segmented file
SEG_FILE = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/processed/HQP079_01_tree_segmented_2.laz")

las = laspy.read(str(SEG_FILE))
xyz = np.column_stack([las.x, las.y, las.z])
tree_ids = las.tree_id

print(f" Cargados {len(xyz):,} points")
print(f"  Árboles únicos: {len(np.unique(tree_ids[tree_ids >= 0]))}")

# Rebuild TreeInfo from data
unique_ids = np.unique(tree_ids[tree_ids >= 0])
tree_info = []
for tid in unique_ids:
    mask = tree_ids == tid
    pts = xyz[mask]
    tree_info.append(TreeInfo(
        tree_id=int(tid),
        centroid=np.mean(pts, axis=0),
        n_points=int(np.sum(mask)),
        height_max=float(np.max(pts[:, 2])),
        height_min=float(np.min(pts[:, 2])),
        axis_direction=np.array([0, 0, 1]),
        axis_deviation_deg=0.0
    ))

seg_result = TreeSegmentationResult(
    tree_ids=tree_ids,
    n_trees=len(tree_info),
    unassigned_count=int(np.sum(tree_ids == -1)),
    tree_info=tree_info
)

# Export locations
locations_file = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/processed/HQP079_01_tree_locations_2.txt")
tree_locations = export_tree_locations(seg_result, locations_file)

print(f"\n Exportado: {locations_file.name}")
print(f"  {len(tree_locations)} locations")
print(f"  En CloudCompare: File > Open > ASCII cloud")

### 7.3 Visualización por Árbol (Open3D)

In [ ]:
import open3d as o3d
import numpy as np

# Generar colores únicos por árbol
np.random.seed(42)
n_trees = seg_result.n_trees + 1  # +1 para no asignados
tree_colors = np.random.rand(n_trees, 3)
tree_colors[0] = [0.5, 0.5, 0.5]  # Gris para no asignados (si tree_id == -1)

# Asignar colores
point_colors = np.zeros((len(veg_normalized), 3))
for i, tid in enumerate(seg_result.tree_ids):
    if tid >= 0:
        point_colors[i] = tree_colors[tid + 1]
    else:
        point_colors[i] = tree_colors[0]

# Crear nube
pcd_seg = o3d.geometry.PointCloud()
pcd_seg.points = o3d.utility.Vector3dVector(veg_normalized)
pcd_seg.colors = o3d.utility.Vector3dVector(point_colors)

print(f"Nube segmentada: {len(pcd_seg.points):,} points, {seg_result.n_trees} árboles")

In [ ]:
o3d.visualization.draw_geometries([pcd_seg], window_name=f"Segmentación: {seg_result.n_trees} árboles", width=1280, height=720)

---
## 8. Próximos Pasos

- **Brick 8:** Análisis por árbol (DBH, height, sweep, branches)
- **Brick 9:** Detección de forks
- **Brick 10:** Clasificación HQP de branches y spikes

## Phase 5 - Workflow Unificado (No Destructivo)

Esta seccion agrega un flujo reproducible para: construir banco de entrenamiento, entrenar clasificador RF y aplicar inferencia sobre un LAS/LAZ.
Las celdas anteriores se mantienen sin cambios para comparacion.


In [ ]:
from pathlib import Path
import json
import pandas as pd

from src.core import (
    build_training_bank,
    reports_to_dataframe,
    split_training_bank_by_plot,
    save_training_splits,
    load_training_data_from_dataframe,
    train_and_evaluate_classifier,
    save_classifier_bundle,
    apply_understory_classifier_to_las,
)

PH5_OUTPUT_DIR = Path('outputs/phase5')
PH5_MODEL_DIR = PH5_OUTPUT_DIR / 'models'
PH5_SPLIT_DIR = PH5_OUTPUT_DIR / 'splits'
PH5_BANK_PATH = PH5_OUTPUT_DIR / 'training_bank.csv'
PH5_REPORTS_PATH = PH5_OUTPUT_DIR / 'training_bank_reports.csv'
PH5_MODEL_PATH = PH5_MODEL_DIR / 'understory_rf_phase5.pkl'
PH5_METRICS_PATH = PH5_MODEL_DIR / 'understory_rf_phase5_metrics.json'
PH5_IMPORTANCE_PATH = PH5_MODEL_DIR / 'understory_rf_phase5_feature_importances.csv'
PH5_INFERENCE_OUTPUT = PH5_OUTPUT_DIR / 'inference' / 'veg_ml_phase5.laz'

PH5_FEATURES = ['verticality', 'linearity', 'sphericity', 'dist_to_ground']
PH5_LABEL_FIELD = 'training_label'
PH5_GROUP_COL = 'plot_id'
PH5_TRAINING_INPUTS = ['notebooks']  # labelled .las/.laz files
PH5_INFERENCE_CANDIDATES = [Path('outputs/vegetation_normalized.laz'), Path('notebooks/trees.laz')]
PH5_INFERENCE_INPUT = next((p for p in PH5_INFERENCE_CANDIDATES if p.exists()), PH5_INFERENCE_CANDIDATES[0])

PH5_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PH5_MODEL_DIR.mkdir(parents=True, exist_ok=True)
PH5_SPLIT_DIR.mkdir(parents=True, exist_ok=True)

def _metric_to_dict(metric_obj):
    data = metric_obj.__dict__.copy()
    data['confusion_matrix'] = metric_obj.confusion_matrix.tolist()
    return data

print('Phase 5 configured')
print(f'- training inputs: {PH5_TRAINING_INPUTS}')
print(f'- inference input: {PH5_INFERENCE_INPUT}')


In [ ]:
bank_df, bank_result = build_training_bank(
    inputs=PH5_TRAINING_INPUTS,
    output_path=PH5_BANK_PATH,
    label_field=PH5_LABEL_FIELD,
)

print('Training bank summary')
print(f'- files_scanned: {bank_result.n_files_scanned}')
print(f'- files_usable: {bank_result.n_files_usable}')
print(f'- points_total: {bank_result.n_points_total:,}')
print(f'- points_kept: {bank_result.n_points_kept:,}')
print(f'- bank_output: {bank_result.output_path}')

reports_df = reports_to_dataframe(bank_result.reports)
reports_df.to_csv(PH5_REPORTS_PATH, index=False)
print(f'- reports_output: {PH5_REPORTS_PATH}')

if bank_df.empty:
    ph5_splits = None
    print('No hay datos etiquetados utilizables. Entrenamiento omitido.')
else:
    ph5_splits = split_training_bank_by_plot(
        bank_df,
        train_ratio=0.70,
        val_ratio=0.15,
        test_ratio=0.15,
        seed=42,
        group_col=PH5_GROUP_COL,
    )
    split_paths = save_training_splits(ph5_splits, PH5_SPLIT_DIR, file_format='csv')
    for split_name, split_df in ph5_splits.items():
        print(f'- {split_name}: {len(split_df):,} -> {split_paths[split_name]}')


In [ ]:
ph5_classifier = None
ph5_metrics = None

if ph5_splits is None:
    print('Entrenamiento omitido por falta de datos etiquetados.')
else:
    train_data = load_training_data_from_dataframe(ph5_splits['train'], feature_names=PH5_FEATURES, label_col='label')
    val_data = None if ph5_splits['val'].empty else load_training_data_from_dataframe(ph5_splits['val'], feature_names=PH5_FEATURES, label_col='label')
    test_data = None if ph5_splits['test'].empty else load_training_data_from_dataframe(ph5_splits['test'], feature_names=PH5_FEATURES, label_col='label')

    ph5_classifier, ph5_metrics = train_and_evaluate_classifier(
        training_data=train_data,
        validation_data=val_data,
        test_data=test_data,
        probability_threshold=0.5,
        n_estimators=300,
        max_depth=14,
        random_state=42,
        verbose=True,
    )

    save_classifier_bundle(
        classifier=ph5_classifier,
        filepath=PH5_MODEL_PATH,
        feature_names=PH5_FEATURES,
        metadata={
            'threshold': 0.5,
            'feature_names': PH5_FEATURES,
            'split_sizes': {k: int(len(v)) for k, v in ph5_splits.items()},
        },
    )

    metrics_payload = {name: _metric_to_dict(m) for name, m in ph5_metrics.items()}
    PH5_METRICS_PATH.write_text(json.dumps(metrics_payload, indent=2), encoding='utf-8')

    importance_df = pd.DataFrame({
        'feature': PH5_FEATURES,
        'importance': ph5_classifier.feature_importances_,
    }).sort_values('importance', ascending=False)
    importance_df.to_csv(PH5_IMPORTANCE_PATH, index=False)

    print('Training completed')
    print(f'- model: {PH5_MODEL_PATH}')
    print(f'- metrics: {PH5_METRICS_PATH}')
    print(f'- feature_importance: {PH5_IMPORTANCE_PATH}')
    for split_name, metric in ph5_metrics.items():
        print(f'- {split_name}: acc={metric.accuracy:.3f}, bal_acc={metric.balanced_accuracy:.3f}, f1_under={metric.f1_understory:.3f}, f1_tree={metric.f1_tree:.3f}')


In [ ]:
if not PH5_MODEL_PATH.exists():
    print(f'Modelo no encontrado: {PH5_MODEL_PATH}. Ejecuta la celda de entrenamiento primero.')
elif not PH5_INFERENCE_INPUT.exists():
    print(f'Input de inferencia no encontrado: {PH5_INFERENCE_INPUT}')
else:
    ph5_inference_result = apply_understory_classifier_to_las(
        input_path=PH5_INFERENCE_INPUT,
        model_path=PH5_MODEL_PATH,
        output_path=PH5_INFERENCE_OUTPUT,
        slice_height=3.5,
        backend='voxel',
        dist_source='auto',
        probability_threshold=0.5,
        write_computed_features=False,
        update_classification=False,
        compress=True,
        verbose=True,
    )
    print('Inferencia completada')
    print(f'- output: {ph5_inference_result.output_path}')
    print(f'- points: {ph5_inference_result.n_points:,}')
    print(f'- tree: {ph5_inference_result.n_tree:,}')
    print(f'- understory: {ph5_inference_result.n_understory:,}')
